# Bella 01 — Dataset Preparation Pipeline

This notebook inventories attached Kaggle datasets, converts supported conversational/code-QA records into a consistent chat format, removes exact duplicate examples, separates explicit test/evaluation files from training, creates a deterministic validation split, and writes a manifest.

**Safety:** this notebook prepares data only. It does not fine-tune or modify the BELLA.01 model. Review dataset licenses, provenance, and sample quality before training. Hugging Face publishing is disabled by default.

In [ ]:
from pathlib import Path
import os, json, hashlib, sqlite3, csv, re, sys
from collections import Counter, defaultdict
from datetime import datetime, timezone

INPUT_ROOT = Path('/kaggle/input/datasets')
OUTPUT_ROOT = Path('/kaggle/working/bella_training_dataset')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# None = include all attached dataset folders. For a controlled first run,
# set this to a list of exact folder names under /kaggle/input/datasets.
DATASET_ALLOWLIST = None
MAX_FILE_SIZE_GB = 2.0
CSV_CHUNK_ROWS = 5000
VALIDATION_PERCENT = 2
PUBLISH_TO_HF = False  # Change only after reviewing outputs and adding HF_TOKEN in Kaggle Secrets.
HF_DATASET_REPO = 'future-prediction-ai/Bella-Training-Datasets'

if not INPUT_ROOT.exists():
    raise FileNotFoundError(f'Kaggle dataset input directory not found: {INPUT_ROOT}. Attach datasets with Add Input.')

dataset_dirs = sorted(p for p in INPUT_ROOT.iterdir() if p.is_dir())
if DATASET_ALLOWLIST is not None:
    dataset_dirs = [p for p in dataset_dirs if p.name in set(DATASET_ALLOWLIST)]
print(f'Input datasets selected: {len(dataset_dirs)}')
for p in dataset_dirs:
    print(' -', p.name)


## 1. Inventory attached files
The inventory is written to the output folder. Unsupported file types are recorded but not treated as training text.

In [ ]:
inventory = []
for dataset_dir in dataset_dirs:
    for file_path in dataset_dir.rglob('*'):
        if not file_path.is_file():
            continue
        try:
            size = file_path.stat().st_size
            inventory.append({
                'dataset': dataset_dir.name,
                'relative_path': str(file_path.relative_to(dataset_dir)),
                'extension': file_path.suffix.lower(),
                'size_bytes': size,
                'size_mb': round(size / 1024**2, 3),
                'status': 'TOO_LARGE' if size > MAX_FILE_SIZE_GB * 1024**3 else 'FOUND'
            })
        except OSError as exc:
            inventory.append({'dataset':dataset_dir.name,'relative_path':str(file_path),'extension':file_path.suffix.lower(),'size_bytes':None,'size_mb':None,'status':f'ERROR: {exc}'})

import pandas as pd
inventory_df = pd.DataFrame(inventory)
inventory_path = OUTPUT_ROOT / 'bella_file_inventory.csv'
inventory_df.to_csv(inventory_path, index=False)
print('Files inventoried:', len(inventory_df))
if not inventory_df.empty:
    print(inventory_df.groupby(['dataset','extension']).size().to_string())
print('Saved:', inventory_path)


## 2. Normalize supported records
Supported inputs: JSONL/NDJSON, JSON, CSV, and Parquet. Records are converted only when a usable user/assistant exchange or conversation is detected. Raw code, metadata, and records with no answer are skipped rather than guessed.

In [ ]:
def clean_text(value):
    if value is None:
        return ''
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return str(value).strip()

def role_name(value):
    role = str(value or '').strip().lower()
    mapping = {'human':'user','user':'user','prompt':'user','instruction':'user',
               'gpt':'assistant','assistant':'assistant','bot':'assistant','completion':'assistant',
               'system':'system','tool':'tool','function':'tool'}
    return mapping.get(role)

def normalize_record(record):
    if not isinstance(record, dict):
        return None
    # Common chat format: {messages:[{role, content}, ...]}
    messages = record.get('messages')
    if isinstance(messages, list):
        normalized = []
        for item in messages:
            if not isinstance(item, dict):
                continue
            role = role_name(item.get('role') or item.get('from') or item.get('speaker'))
            content = clean_text(item.get('content', item.get('value', item.get('text'))))
            if role and content:
                normalized.append({'role':role,'content':content})
        if len(normalized) >= 2 and any(m['role']=='assistant' for m in normalized):
            return normalized
    # ShareGPT-style format: {conversations:[{from, value}, ...]}
    conversations = record.get('conversations')
    if isinstance(conversations, list):
        normalized = []
        for item in conversations:
            if not isinstance(item, dict):
                continue
            role = role_name(item.get('from') or item.get('role') or item.get('speaker'))
            content = clean_text(item.get('value', item.get('content', item.get('text'))))
            if role and content:
                normalized.append({'role':role,'content':content})
        if len(normalized) >= 2 and any(m['role']=='assistant' for m in normalized):
            return normalized
    # Instruction / prompt completion pairs
    user_text = record.get('prompt') or record.get('instruction') or record.get('question') or record.get('query')
    if user_text is not None:
        extra = clean_text(record.get('input'))
        if extra:
            user_text = clean_text(user_text) + '\n\n' + extra
        assistant_text = record.get('completion') or record.get('response') or record.get('output') or record.get('answer')
        if clean_text(user_text) and clean_text(assistant_text):
            return [{'role':'user','content':clean_text(user_text)}, {'role':'assistant','content':clean_text(assistant_text)}]
    return None

def iter_file_records(path):
    ext = path.suffix.lower()
    if ext in ('.jsonl','.ndjson'):
        with path.open('r', encoding='utf-8', errors='replace') as f:
            for line_no, line in enumerate(f, 1):
                if line.strip():
                    try:
                        yield json.loads(line)
                    except json.JSONDecodeError:
                        counters['malformed_json'] += 1
    elif ext == '.json':
        with path.open('r', encoding='utf-8', errors='replace') as f:
            data = json.load(f)
        if isinstance(data, list):
            yield from data
        elif isinstance(data, dict):
            found = False
            for key in ('data','records','examples','train','validation'):
                if isinstance(data.get(key), list):
                    yield from data[key]
                    found = True
            if not found:
                yield data
    elif ext == '.csv':
        for chunk in pd.read_csv(path, chunksize=CSV_CHUNK_ROWS, on_bad_lines='skip'):
            yield from chunk.to_dict(orient='records')
    elif ext in ('.parquet','.pq'):
        import pyarrow.parquet as pq
        parquet_file = pq.ParquetFile(path)
        for batch in parquet_file.iter_batches(batch_size=1000):
            yield from batch.to_pylist()

print('Normalization functions ready.')


## 3. Build train, validation, and held-out files
Files whose paths clearly indicate test/evaluation data are kept out of training. Remaining examples are deduplicated by normalized conversation content and assigned deterministically to train/validation.

In [ ]:
train_path = OUTPUT_ROOT / 'train.jsonl'
validation_path = OUTPUT_ROOT / 'validation.jsonl'
heldout_path = OUTPUT_ROOT / 'heldout.jsonl'
db_path = Path('/kaggle/temp/bella_seen_hashes.sqlite')
db_path.parent.mkdir(parents=True, exist_ok=True)
if db_path.exists():
    db_path.unlink()
conn = sqlite3.connect(db_path)
conn.execute('CREATE TABLE seen (digest TEXT PRIMARY KEY)')

counters = Counter()
by_dataset = defaultdict(Counter)
sample_records = []
supported_exts = {'.jsonl','.ndjson','.json','.csv','.parquet','.pq'}
eval_pattern = re.compile(r'(^|[/_.-])(test|tests|eval|evaluation|valid|validation|dev)([/_.-]|$)', re.I)

with train_path.open('w', encoding='utf-8') as train_f, \
     validation_path.open('w', encoding='utf-8') as val_f, \
     heldout_path.open('w', encoding='utf-8') as heldout_f:
    for dataset_dir in dataset_dirs:
        for file_path in sorted(dataset_dir.rglob('*')):
            if not file_path.is_file() or file_path.suffix.lower() not in supported_exts:
                continue
            try:
                if file_path.stat().st_size > MAX_FILE_SIZE_GB * 1024**3:
                    counters['skipped_too_large'] += 1
                    continue
                relative = str(file_path.relative_to(dataset_dir))
                is_heldout = bool(eval_pattern.search(relative))
                for raw_record in iter_file_records(file_path):
                    counters['records_seen'] += 1
                    normalized = normalize_record(raw_record)
                    if not normalized:
                        counters['records_not_training_pairs'] += 1
                        continue
                    canonical = json.dumps(normalized, ensure_ascii=False, sort_keys=True, separators=(',',':'))
                    digest = hashlib.sha256(canonical.encode('utf-8')).hexdigest()
                    try:
                        conn.execute('INSERT INTO seen(digest) VALUES (?)', (digest,))
                    except sqlite3.IntegrityError:
                        counters['duplicates_removed'] += 1
                        continue
                    output_record = {'messages':normalized,'source_dataset':dataset_dir.name,'source_file':relative}
                    line = json.dumps(output_record, ensure_ascii=False) + '\n'
                    if is_heldout:
                        heldout_f.write(line)
                        counters['heldout_examples'] += 1
                        by_dataset[dataset_dir.name]['heldout'] += 1
                    else:
                        # Stable split, independent of run order.
                        bucket = int(digest[:8], 16) % 100
                        if bucket < VALIDATION_PERCENT:
                            val_f.write(line)
                            counters['validation_examples'] += 1
                            by_dataset[dataset_dir.name]['validation'] += 1
                        else:
                            train_f.write(line)
                            counters['training_examples'] += 1
                            by_dataset[dataset_dir.name]['train'] += 1
                    counters['unique_examples'] += 1
                    if len(sample_records) < 8:
                        sample_records.append(output_record)
            except Exception as exc:
                counters['files_with_errors'] += 1
                by_dataset[dataset_dir.name]['file_errors'] += 1
                print(f'FILE ERROR [{dataset_dir.name}/{file_path.name}]: {type(exc).__name__}: {exc}')
            if counters['records_seen'] and counters['records_seen'] % 100000 == 0:
                conn.commit()
                print('Progress:', dict(counters))
    conn.commit()
conn.close()

print('\nFINAL COUNTS')
for key, value in counters.most_common():
    print(f'{key}: {value:,}')
print('\nSAMPLE NORMALIZED RECORDS')
for item in sample_records:
    print(json.dumps(item, ensure_ascii=False)[:1000])


## 4. Write manifest and verify outputs
Check that files exist, are non-empty where expected, and that each JSONL row contains messages. This is a structural check, not a guarantee that the source content is accurate or licensed for training.

In [ ]:
def count_jsonl(path):
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

output_counts = {
    'train': count_jsonl(train_path),
    'validation': count_jsonl(validation_path),
    'heldout': count_jsonl(heldout_path),
}
for name, path in [('train',train_path),('validation',validation_path),('heldout',heldout_path)]:
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            for line_no, line in enumerate(f, 1):
                if line.strip():
                    row = json.loads(line)
                    assert isinstance(row.get('messages'), list) and len(row['messages']) >= 2, f'Invalid {name} row {line_no}'
                    break

manifest = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'input_root': str(INPUT_ROOT),
    'selected_datasets': [p.name for p in dataset_dirs],
    'format': 'chat messages JSONL',
    'counts': dict(counters),
    'output_counts': output_counts,
    'outputs': {name: str(path) for name,path in [('train',train_path),('validation',validation_path),('heldout',heldout_path)]},
    'notes': [
        'Only recognized conversational or prompt/answer records were converted.',
        'Exact duplicate normalized conversations were removed.',
        'Explicit test/evaluation/validation/dev paths were excluded from train and validation.',
        'Dataset licenses and content safety require human review before model training.'
    ]
}
manifest_path = OUTPUT_ROOT / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
dataset_report = pd.DataFrame([{'dataset':name, **dict(values)} for name,values in by_dataset.items()]).fillna(0)
dataset_report_path = OUTPUT_ROOT / 'dataset_report.csv'
dataset_report.to_csv(dataset_report_path, index=False)

print('\nOUTPUT SUMMARY')
print(json.dumps(output_counts, indent=2))
print('Manifest:', manifest_path)
print('Dataset report:', dataset_report_path)
print('Output directory:', OUTPUT_ROOT)
if output_counts['train'] == 0:
    print('WARNING: No training examples were produced. Review the inventory and add a dataset-specific normalizer for the source schema.')
else:
    print('STRUCTURAL CHECK PASSED. Review samples and licensing before training or publishing.')


## 5. Optional: publish prepared dataset to Hugging Face
This is deliberately disabled by default. First inspect the generated files and confirm the repository target. To publish, add a Kaggle Secret named `HF_TOKEN`, set `PUBLISH_TO_HF = True` in the configuration cell, then run this cell. It uploads the prepared output folder to the existing private dataset repository.

In [ ]:
if not PUBLISH_TO_HF:
    print('Hugging Face upload is disabled. Review OUTPUT_ROOT before enabling it.')
else:
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import HfApi
        hf_token = UserSecretsClient().get_secret('HF_TOKEN')
        if not hf_token:
            raise RuntimeError('Kaggle Secret HF_TOKEN is empty or missing.')
        api = HfApi(token=hf_token)
        api.create_repo(repo_id=HF_DATASET_REPO, repo_type='dataset', private=True, exist_ok=True)
        api.upload_folder(
            repo_id=HF_DATASET_REPO,
            repo_type='dataset',
            folder_path=str(OUTPUT_ROOT),
            commit_message='Publish validated Bella training dataset build'
        )
        print('Upload completed:', f'https://huggingface.co/datasets/{HF_DATASET_REPO}')
    except Exception as exc:
        print('Upload failed:', type(exc).__name__, str(exc))
        raise
